In [1]:
import torch

In [40]:
def sample_domain():
    X = torch.tensor(
           [[0.9250, 0.9151, 0.32],
            [0.0815, 0.3014, 0.16],
            [0.2055, 0.2310, 0.67],
            [0.9597, 0.1719, 0.75]]
    )
    return X

r2 = lambda x: torch.sum(x**2, dim=1).unsqueeze(dim=1)
model = lambda x: torch.cos(r2(x))
model_grad =   lambda x: -2 * x * torch.sin(r2(x))
model_grad_2 = lambda x: -2 * torch.sin(r2(x)) - 4 * x**2 * torch.cos(r2(x))

X = sample_domain()
X.requires_grad_(True)
u = model(X)
print(X.shape, u.shape)

torch.Size([4, 3]) torch.Size([4, 1])


---

## normal

In [3]:
def compute_derivatives(model, X):
    """
    Compute u, grad u, and laplace u
    X: (batch_size, D) where D = d + 1 (spatial dims + time)
    """
    u = model(X)
    bs, D = X.shape

    # Gradient - spatial & temporatal
    grad_u = torch.autograd.grad(
        inputs=X,
        outputs=u,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]

    # Laplacian - spatial only
    #spatial_laplace_u = torch.zeros((bs,D-1))
    spatial_laplace_u = []
    for i in range(D-1):
        hess_row = torch.autograd.grad(
            inputs=X,
            outputs=grad_u[:,i].sum(),
            grad_outputs=torch.tensor(1.0),
            create_graph=True,
            retain_graph=True
        )[0]
        spatial_laplace_u.append(hess_row[:,i:i+1])
    spatial_laplace_u = torch.cat(spatial_laplace_u, dim=1)

    # shapes: bs x 1, bs x D, bs x D-1
    #return u, grad_u, spatial_laplace_u
    return u, grad_u, spatial_laplace_u

In [4]:
u, grad_u, spatial_laplace_u = compute_derivatives(model, X)
spatial_laplace_u

tensor([[-1.1874, -1.2036],
        [-0.2719, -0.6062],
        [-1.1805, -1.2185],
        [-2.2092, -2.0035]], grad_fn=<CatBackward0>)

In [5]:
spatial_laplace_u.sum(dim=1)

tensor([-2.3910, -0.8781, -2.3990, -4.2127], grad_fn=<SumBackward1>)

In [74]:
laplace_analytic = model_grad_2(X).sum(dim=1)
laplace_analytic

tensor([-4.2495, -1.2253, -4.9709, -6.3392], grad_fn=<SumBackward1>)

# HTE

In [202]:
### V1
X = sample_domain()
X.requires_grad_(True)
N,D = X.shape
d = D-1
print(f"d = {d}")

def value_point(point):
    return model(point.unsqueeze(dim=0)).squeeze()

from torch.func import jacrev, jvp
def laplace_hutchinson_point(point):
    num_vectors = 1000
    vectors = torch.randn(num_vectors, len(point))
    grad = lambda point: jacrev(value_point)(point)
    jvp_f = lambda v: torch.dot(v, jvp(grad, (point,), (v,))[1])
    return torch.sum(torch.vmap(jvp_f)(vectors))/num_vectors

def laplace_hutchinson(points):
    return torch.vmap(laplace_hutchinson_point, randomness="same")(points)

#laplace_hutchinson_point(X[0])
lh = laplace_hutchinson(X)
lh - laplace_analytic

d = 2


tensor([0.1397, 0.0414, 0.1719, 0.2179], grad_fn=<SubBackward0>)

---

In [215]:
def compute_hte(randomness, get_sampled_vectors):
    X = sample_domain()
    X.requires_grad_(True)
    N,D = X.shape
    d = D-1

    def value_point(point):
        return model(point.unsqueeze(dim=0)).squeeze()

    from torch.func import jacrev, jvp
    def laplace_hutchinson_point(point):
        num_vectors = 50
        vectors = get_sampled_vectors(num_vectors, len(point))
        grad = lambda point: jacrev(value_point)(point)
        jvp_f = lambda v: torch.dot(v, jvp(grad, (point,), (v,))[1])
        return torch.sum(torch.vmap(jvp_f)(vectors))/num_vectors

    def laplace_hutchinson(points):
        return torch.vmap(laplace_hutchinson_point, randomness=randomness)(points)

    #laplace_hutchinson_point(X[0])
    lh = laplace_hutchinson(X)
    return lh - laplace_analytic

In [210]:
### V2
randomness = ["same", "different"]
get_sampled_vectors = [
    lambda n1,n2: 2.0*torch.randint(0,2,(n1,n2))-1,
    lambda n1,n2: torch.randn(n1,n2)
]
randomness = randomness[0]
get_sampled_vectors = get_sampled_vectors[0]

n_trials = 100
errs = torch.zeros(X.shape[0], n_trials)
for i in range(n_trials):
    err = compute_hte(randomness, get_sampled_vectors).abs()
    errs[:,i] = err

torch.mean(errs, dim=1), torch.std(errs, dim=1)

(tensor([0.0417, 0.0118, 0.0414, 0.0095], grad_fn=<MeanBackward1>),
 tensor([0.0315, 0.0083, 0.0281, 0.0067], grad_fn=<StdBackward0>))

In [216]:
### V2
randomness = ["same", "different"]
get_sampled_vectors = [
    lambda n1,n2: 2.0*torch.randint(0,2,(n1,n2))-1,
    lambda n1,n2: torch.randn(n1,n2)
]

n_trials = 1000
for r in randomness:
    for j, gsv in enumerate(get_sampled_vectors):
        errs = torch.zeros(X.shape[0], n_trials)
        for i in range(n_trials):
            err = compute_hte(r, gsv).abs()
            errs[:,i] = err
        print(r, ["radek", "gauss"][j])
        print("  mean =", torch.mean(errs, dim=1))
        print("  std =", torch.std(errs, dim=1))

same radek
  mean = tensor([0.1845, 0.0486, 0.1647, 0.0392], grad_fn=<MeanBackward1>)
  std = tensor([0.1371, 0.0361, 0.1214, 0.0288], grad_fn=<StdBackward0>)
same gauss
  mean = tensor([0.4449, 0.1266, 0.5102, 0.5703], grad_fn=<MeanBackward1>)
  std = tensor([0.3438, 0.1006, 0.4033, 0.4495], grad_fn=<StdBackward0>)
different radek
  mean = tensor([0.1965, 0.0498, 0.1595, 0.0388], grad_fn=<MeanBackward1>)
  std = tensor([0.1431, 0.0393, 0.1255, 0.0301], grad_fn=<StdBackward0>)
different gauss
  mean = tensor([0.4631, 0.1297, 0.5339, 0.5646], grad_fn=<MeanBackward1>)
  std = tensor([0.3555, 0.0996, 0.3857, 0.4302], grad_fn=<StdBackward0>)


=> 'radek' is the best!, doesn't really matter whether 'same' of 'different'